# OsteoClassAI v2 · 10 Detector de articulación, paso 1: datos

Usa `KneeXrayData.zip` de Chen (doi:10.17632/56rmx5bjcr.1, CC BY 4.0), ya guardado en `MyDrive/OsteoClassAI/datos_externos/`. La parte `DetKneeData/H5` trae una radiografía completa por paciente (256 × 320) con la caja de cada rodilla, en la **misma partición por pacientes** que los recortes de clasificación (2889 / 413 / 828), así que entrenar con `train` no toca la prueba.

Este paso:
1. convierte los `.h5` en PNG + `index.csv`;
2. mide cómo se relaciona cada caja con el recorte de 224 px que usaron A, B, C y D (lado de la imagen, espejo, margen y desplazamiento), buscando el recorte dentro de la radiografía en 200 pacientes de entrenamiento.

*Entorno de ejecución:* CPU sirve (~5 min, la mayor parte copiando el zip).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . h5py opencv-python-headless

In [ ]:
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/KneeXrayData.zip'
!test -f /content/KneeXrayData.zip || cp "$ZIP" /content/KneeXrayData.zip
!python -m osteoclass.detdata --zip /content/KneeXrayData.zip --out outputs/det --calibrate 200
!mkdir -p /content/drive/MyDrive/OsteoClassAI/det && cp -r outputs/det/. /content/drive/MyDrive/OsteoClassAI/det/

## Revisión visual: cajas de Chen y recorte encontrado (6 pacientes)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, matplotlib.patches as pa, cv2
idx = pd.read_csv('outputs/det/index.csv'); m = pd.read_csv('outputs/det/calibration_matches.csv')
best = m.loc[m.groupby(['patient', 'side']).score.idxmax()]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, p in zip(ax.flat, best.patient.drop_duplicates().head(6)):
    img = cv2.imread(f'outputs/det/images/{p}.png', 0); a.imshow(img, cmap='gray'); a.set_title(str(p)); a.axis('off')
    for _, b in idx[idx.patient == p].iterrows():
        a.add_patch(pa.Rectangle((b.x1, b.y1), b.x2 - b.x1, b.y2 - b.y1, fill=False, ec='lime', lw=1.5))
    for _, r in best[best.patient == p].iterrows():
        bx = idx[(idx.patient == p) & (idx.box == r.box)].iloc[0]
        cx = (bx.x1 + bx.x2) / 2 + r.dx * (bx.x2 - bx.x1); cy = (bx.y1 + bx.y2) / 2 + r.dy * (bx.y2 - bx.y1)
        a.add_patch(pa.Rectangle((cx - r.crop_size / 2, cy - r.crop_size / 2), r.crop_size, r.crop_size,
                                 fill=False, ec='red', lw=1.5, ls='--'))
        a.text(cx, cy - r.crop_size / 2 - 3, f'{r.side} {r.score:.2f}', color='red', ha='center')
plt.tight_layout(); plt.show()